In [10]:
import numpy as np
from collections import defaultdict

class KneserNeyLanguageModel:
    def __init__(self, order=5, discount=0.75):
        """
        Initializes the Kneser-Ney LM.
        :param order: The maximum n-gram order (e.g., 5 for 5-gram)
        :param discount: The absolute discount value 'd' (usually between 0.5 and 0.75)
        """
        self.order = order
        self.discount = discount
        self.vocab = set()
        
        # Dense Sparse Tables (Hash Maps)
        self.counts = defaultdict(lambda: defaultdict(int))          # Raw n-gram counts
        self.C_KN = defaultdict(lambda: defaultdict(int))            # Kneser-Ney pseudo-counts
        self.Denom_KN = defaultdict(lambda: defaultdict(int))        # Denominators for probabilities
        self.Num_Following = defaultdict(lambda: defaultdict(int))   # Unique words following a context (for lambda)

    def _tokenize(self, text):
        """Simple whitespace tokenizer and lowercaser."""
        return text.lower().split()

    def fit(self, corpus):
        """Builds the vocabulary and precomputes all recursive Kneser-Ney tables."""
        # Initialize vocab with special tokens
        self.vocab = {'<UNK>', '<s>', '</s>'}
        tokenized_corpus = []
        
        for sentence in corpus:
            tokens = self._tokenize(sentence)
            self.vocab.update(tokens)
            tokenized_corpus.append(tokens)

        # 1. Count raw n-grams of all sizes up to `self.order`
        for tokens in tokenized_corpus:
            padded_tokens = ['<s>'] * (self.order - 1) + tokens + ['</s>']
            for n in range(1, self.order + 1):
                for i in range(len(padded_tokens) - n + 1):
                    ngram = tuple(padded_tokens[i : i + n])
                    self.counts[n][ngram] += 1

        # 2. Build Kneser-Ney pseudo-counts (C_KN)
        # Highest order uses raw training counts
        for ngram, count in self.counts[self.order].items():
            self.C_KN[self.order][ngram] = count

        # Lower orders use Continuation Counts 
        # C_KN(ngram) = |{ w : count(w + ngram) > 0 }|
        for k in range(self.order - 1, 0, -1):
            preceding_words = defaultdict(set)
            for k_plus_1_gram, count in self.counts[k + 1].items():
                if count > 0:
                    w_prev = k_plus_1_gram[0]
                    ngram_tail = k_plus_1_gram[1:]
                    preceding_words[ngram_tail].add(w_prev)
            
            for ngram, w_set in preceding_words.items():
                self.C_KN[k][ngram] = len(w_set)

        # 3. Build Denominators and Num_Following (Lambda components)
        for k in range(self.order):
            for k_plus_1_gram, count in self.C_KN[k + 1].items():
                if count > 0:
                    context = k_plus_1_gram[:-1]
                    self.Denom_KN[k][context] += count
                    self.Num_Following[k][context] += 1

    def get_probability(self, word, context):
        """Calculates P(word | context) dynamically replacing OOVs."""
        word = word if word in self.vocab else '<UNK>'
        context = tuple([w if w in self.vocab else '<UNK>' for w in context])

        # Truncate context if it exceeds the maximum supported length
        if len(context) > self.order - 1:
            context = context[-(self.order - 1):]

        return self._recursive_kn(word, context)

    def _recursive_kn(self, word, context):
        """Recursively calculates absolute discounting and back-off."""
        k = len(context)

        if k == 0:
            # Base Case (Unigram): P(w) = max(N_1+(.w) - d, 0) / N_1+(..) + lambda * (1 / |V|)
            c_kn = self.C_KN[1].get((word,), 0)
            denom = self.Denom_KN[0].get((), 0)
            num_following = self.Num_Following[0].get((), 0)

            if denom == 0:
                return 1.0 / len(self.vocab) # Extreme fallback

            prob = max(c_kn - self.discount, 0.0) / denom
            gamma = (self.discount / denom) * num_following
            
            # Backoff to uniform distribution over the whole vocabulary
            prob += gamma * (1.0 / len(self.vocab))
            return prob
        
        else:
            # Recursive Case (n-gram)
            ngram = context + (word,)
            c_kn = self.C_KN[k + 1].get(ngram, 0)
            denom = self.Denom_KN[k].get(context, 0)
            num_following = self.Num_Following[k].get(context, 0)

            # If the context is completely unseen, drop the first word and backoff 100%
            if denom == 0:
                return self._recursive_kn(word, context[1:])

            prob = max(c_kn - self.discount, 0.0) / denom
            gamma = (self.discount / denom) * num_following
            
            # Interpolate with the lower order probability
            prob += gamma * self._recursive_kn(word, context[1:])
            return prob

    def calculate_sentence_perplexity(self, sentence):
        """Calculates 2^(- average log2 probability) for a sentence."""
        tokens = ['<s>'] * (self.order - 1) + self._tokenize(sentence) + ['</s>']
        log_prob_sum = 0.0
        N = 0

        for i in range(self.order - 1, len(tokens)):
            context = tuple(tokens[i - self.order + 1 : i])
            word = tokens[i]
            
            p = self.get_probability(word, context)
            log_prob_sum += np.log2(p)
            N += 1

        return np.power(2, -log_prob_sum / N)

# ==========================================
# MENTOR'S SANITY CHECK / TESTING BLOCK
# ==========================================
if __name__ == "__main__":
    # Mini-Corpus containing recursive structure patterns for testing
    toy_corpus = [
        "The quick brown fox jumps over the lazy dog",
        "A quick brown fox running along the river bank",
        "The dog barked loudly at the quick brown fox",
        "Never run near the river bank in the dark",
        "The quick brown fox leaped high in the air"
    ]

    print("Initializing 5-gram Kneser-Ney Language Model...")
    lm = KneserNeyLanguageModel(order=5, discount=0.75)
    lm.fit(toy_corpus)

    # Confirm structural sanity of the vocabulary space
    print(f"Total Model Vocabulary Size: {len(lm.vocab)}")

    # Run sanity checks to verify that probabilities sum to 1.0
    test_context = ("the", "quick", "brown", "fox")
    prob_sum = 0.0
    for vocab_word in lm.vocab:
        prob_sum += lm.get_probability(vocab_word, test_context)

    print(f"Sum of P(w | 'the quick brown fox') over all vocab: {prob_sum:.6f} (Expected: ~1.0)")

    # Assess translation and context completion likelihoods
    word_candidates = ["jumps", "leaped", "barked", "river"]
    print("\nConditional Likelihood Predictions for context 'the quick brown fox':")
    for w in word_candidates:
        p = lm.get_probability(w, test_context)
        print(f" - P({w:<10} | context) = {p:.6f}")

    # Evaluate testing set Perplexity
    test_sentence = "The quick brown fox jumps"
    perplexity = lm.calculate_sentence_perplexity(test_sentence)
    print(f"\nSentence: '{test_sentence}' -> Perplexity: {perplexity:.4f}")

    out_of_vocab_test = "The fast magenta unicorn jumps"
    perplexity_oov = lm.calculate_sentence_perplexity(out_of_vocab_test)
    print(f"Sentence: '{out_of_vocab_test}' (contains OOVs) -> Perplexity: {perplexity_oov:.4f}")

Initializing 5-gram Kneser-Ney Language Model...
Total Model Vocabulary Size: 27
Sum of P(w | 'the quick brown fox') over all vocab: 1.000000 (Expected: ~1.0)

Conditional Likelihood Predictions for context 'the quick brown fox':
 - P(jumps      | context) = 0.199827
 - P(leaped     | context) = 0.199827
 - P(barked     | context) = 0.008095
 - P(river      | context) = 0.008095

Sentence: 'The quick brown fox jumps' -> Perplexity: 2.7234
Sentence: 'The fast magenta unicorn jumps' (contains OOVs) -> Perplexity: 29.5190
